> 在线讲义：[第 13 章 · 规约、矩阵乘与数值稳定](https://wikieden.github.io/pytorch-inside-out/lessons/13-reduce-matmul.html) · 本书《PyTorch 从里到外》

# 第 13 章 · 规约、矩阵乘与数值稳定

本章回答：`sum(dim=0)` 和 `sum(dim=1)` 到底沿哪个方向加？`@`、`bmm`、`einsum` 怎么选，一次矩阵乘法要算多少次？为什么直接算 `log(sum(exp(x)))` 会得到 inf，`logsumexp` 是怎么躲过去的？为什么 bf16 一路累加 1000 个 1 只得到 256？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/13-reduce-matmul.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §4.4–§4.5；Transformer 讲义 第 6 章 softmax、第 7 章 饱和；Ultra-Scale Playbook §Mixed Precision Training（`distributed-pretraining/materials/text/ultrascale-playbook.md` 第 2821 行起）；本章输出在本机 torch 2.15.0 上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 dim | 压扁那一维 | `reductions()` | [5, 7, 9]、[6, 15]、keepdim (2, 1) |
| 第二步 einsum / FLOPs | 2mkn | `einsum_checks()`、`matmul_flops` | 点积 32；4096³ 是 1.37 × 10¹¹ |
| 第三步 logsumexp | M + log Σ e^{x−M} | `logsumexp_demo()` | 直接算 inf，logsumexp 1002.4076 |
| 第四步 bf16 累加 | 间隔 2 | `bf16_running_sum()` | 逐个加卡在 256；sum 得 4096 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `bf16_running_sum`**（来自 `ch13_reduce_matmul.py`）：下面要用到的小工具。

In [ ]:
def bf16_running_sum(n: int = 1000):
    s = torch.zeros((), dtype=torch.bfloat16)
    one = torch.tensor(1.0, dtype=torch.bfloat16)
    for _ in range(n):
        s = s + one
    return s.item(), torch.ones(4096, dtype=torch.bfloat16).sum().item()

**函数 `einsum_checks`**（来自 `ch13_reduce_matmul.py`）：下面要用到的小工具。

In [ ]:
def einsum_checks():
    A, B = torch.randn(4, 2, 3), torch.randn(4, 3, 5)
    dot = torch.einsum("i,i->", torch.tensor([1.0, 2, 3]), torch.tensor([4.0, 5, 6])).item()
    return dot, torch.allclose(torch.einsum("bij,bjk->bik", A, B), A @ B)

**函数 `logsumexp_demo`**（来自 `ch13_reduce_matmul.py`）：下面要用到的小工具。

In [ ]:
def logsumexp_demo():
    big = torch.tensor([1000.0, 1001.0, 1002.0])
    return torch.exp(big).sum().log().item(), torch.logsumexp(big, 0).item(), torch.softmax(big, 0).tolist()

**函数 `matmul_flops`**（来自 `ch13_reduce_matmul.py`）：下面要用到的小工具。

In [ ]:
def matmul_flops(m: int, k: int, n: int) -> int:
    return 2 * m * k * n

**函数 `reductions`**（来自 `ch13_reduce_matmul.py`）：下面要用到的小工具。

In [ ]:
def reductions():
    t = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    return t.sum(0).tolist(), t.sum(1).tolist(), tuple(t.sum(1, keepdim=True).shape), t.max(-1)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 sum(0) / sum(1) / keepdim 形状 / max：", reductions())

In [ ]:
print("\n第二步 einsum 点积、和 @ 一致：", einsum_checks(), " 4096³ FLOPs =", f"{matmul_flops(4096, 4096, 4096):.3e}")

In [ ]:
print("\n第三步 直接算 / logsumexp / softmax：", logsumexp_demo())

In [ ]:
print("\n第四步 bf16 逐个加 1000 次 / sum 4096 个 1：", bf16_running_sum())

## 3. 代码里才看得到的

1. **`t.max(-1)` 返回两样**：`values` 和 `indices`，不是一个张量。
2. **`torch.sum` 对 bf16 不会卡住**：kernel 内部用 float32 累加；卡住的是 Python 里一次次 `s + one`。

## 4. 自己改一改

- 把 `bf16_running_sum` 改成 float16，看卡在多少。
- `torch.logsumexp(torch.tensor([-1000., -1000.]), 0)`，再和直接算对比。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> dim 是被压扁的那一维，keepdim 留位置；
> 矩阵乘算 2mkn，einsum 两边都有的留下、只在左边的求和；
> logsumexp 先减最大，exp 就不爆；存可以低精度，累加要用 fp32。